### Creating a neural network from scratch

#### Inspiration from micrograd lectures by Andrej Karpathy: https://github.com/karpathy/nn-zero-to-hero/tree/master/lectures/micrograd

### Imports

In [25]:
import math
import matplotlib.pyplot as plt
import numpy as np
import random
import torch

%matplotlib inline

### Classes

In [26]:
class Value:

    def __init__(self, val, _children=()):
        self.val = val
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        
    def __repr__(self):
        return "val={}".format(self.val)
        
    def __add__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
        
        out = Value(self.val + other.val, (self, other))
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
            
        return out
    
    def __sub__(self, other):
        return self + (-other)
    
    def __neg__(self):
        return self * -1
    
    def __mul__(self, other):
        if not isinstance(other, Value):
            other = Value(other)
            
        out = Value(self.val * other.val, (self, other))
        
        def _backward():
            self.grad += other.val * out.grad
            other.grad += self.val * out.grad
        self._backward = _backward
        
        return out
        
    def __truediv__(self, other): # 5 / 2 = 5 * (1/2) = 2.5
        return self * other**-1
    
    def __pow__(self, other):
        out = Value(self.val**other, (self,))
        
        def _backward():
            self.grad += other * (self.val ** (other - 1)) * out.grad
        out._backward = _backward
        
        return out
    
    def exp(self):
        out = Value(math.exp(self.val), (self,))
        
        def _backward():
            self.grad += out.val * out.grad
        out._backward = _backward
            
        return out
    
    def __radd__(self, other):
        return self + other
    
    def sigmoid(self):
        s = 1 / (1 + math.exp(-self.val))
        out = Value(s, (self,))
        
        def _backward():
            self.grad += s * (1-s) * out.grad
        out._backward = _backward
        
        return out
    
    def topo_sort(self, node, res):
        visited = set()        
        
        if node not in visited:
            visited.add(node)
            for child in node._prev:
                self.topo_sort(child, res)
            res.append(node)
        
        return res
    
    def backward(self):        
        topo_list = self.topo_sort(self, [])
        
        # Necessary for results to be non-zero from calculation
        # dL / dL = 1.0
        self.grad = 1.0
        
        # Traverse the nodes backward from the final output
        for node in reversed(topo_list):
            node._backward()

In [27]:
class Neuron:
    
    def __init__(self, input_size):
        self.w = [Value(random.uniform(-1,1)) for _ in range(input_size)]
        self.b = Value(random.uniform(-1,1))
    
    def __call__(self, x):
        y = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        output = y.sigmoid()
        return output
    
    def get_params(self):
        return self.w + [self.b]
        
class Layer:

    def __init__(self, input_size, output_size):
        self.neurons = [Neuron(input_size) for _ in range(output_size)]
    
    def __call__(self, x):
        outputs = [neuron(x) for neuron in self.neurons]
        if len(outputs) == 1:
            return outputs[0]
        else:
            return outputs
    
    def get_params(self):
        params = []
        for neuron in self.neurons:
            for p in neuron.get_params():
                params.append(p)
        
        return params

class Model:
    
    def __init__(self, input_size, layer_sizes):
        dims = [input_size] + layer_sizes
        layers = [Layer(dims[i], dims[i+1]) for i in range(len(dims)-1)]
        self.layers = layers
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def get_params(self):
        params = []
        for layer in self.layers:
            for p in layer.get_params():
                params.append(p)
        
        return params

### Train

In [28]:
def sigmoid(x):
    return 1.0 / (1 + math.exp(-x))

# f(a,b,c,d) = 15.0*a + 7.0*b - 1.5*c + 2.0*d
def f(x):
    y = 15.0*x[0] + 7.0*x[1] - 1.5*x[2] + 2.0*x[3]
    return (sigmoid(y) >= 0.5) * 1

In [29]:
xs = [[random.uniform(-5, 5) for _ in range(4)] for _ in range(32)]
ys = [f(xs[i]) for i in range(len(xs))]

In [30]:
print(ys)

[0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1]


In [31]:
model = Model(4, [8, 8, 1])

In [32]:
print(model([1, 5, 3, 4]).val)

0.8572055752594839


In [33]:
BATCH_SIZE = 4
EPOCHS = 2000
lr = 0.01

for ep in range(1, EPOCHS+1):
    for i in range(0, len(xs), BATCH_SIZE):
        # Forward pass
        xb = xs[i:i+BATCH_SIZE]
        ytb = ys[i:i+BATCH_SIZE]
        
        y_pred = [model(xi) for xi in xb]
        loss = sum((yp - yt)**2 for yt,yp in zip(ytb, y_pred))

        # Backward pass    
        for p in model.get_params():
            # Update gradients to 0.0 so it doesn't accumulate
            p.grad = 0.0
        loss.backward()

        # Update parameters
        for p in model.get_params():
            p.val = p.val - lr*p.grad

    if (ep % 100 == 0):
        print("epoch: {}, loss: {}".format(ep, loss.val))

epoch: 100, loss: 0.4306392838357131
epoch: 200, loss: 0.29331365527514913
epoch: 300, loss: 0.2446072809266313
epoch: 400, loss: 0.21670104417740854
epoch: 500, loss: 0.19165872698549172
epoch: 600, loss: 0.17321060898103793
epoch: 700, loss: 0.1557991942359792
epoch: 800, loss: 0.1397210992352917
epoch: 900, loss: 0.1263026516848253
epoch: 1000, loss: 0.11310573421414326
epoch: 1100, loss: 0.1016177952006436
epoch: 1200, loss: 0.09264690608056299
epoch: 1300, loss: 0.08416986701668715
epoch: 1400, loss: 0.07596268496730865
epoch: 1500, loss: 0.06820102585492854
epoch: 1600, loss: 0.06182305359349668
epoch: 1700, loss: 0.05548351387139597
epoch: 1800, loss: 0.05046223012287093
epoch: 1900, loss: 0.04618600725585396
epoch: 2000, loss: 0.04236374854237529


In [40]:
y_pred = [(model(x).val >= 0.5)*1 for x in xs]

In [45]:
for i in range(len(ys)):
    print("pred: {}; target: {}; correct: {}".format(y_pred[i], ys[i], y_pred[i] == ys[i]))

pred: 0; target: 0; correct: True
pred: 0; target: 0; correct: True
pred: 0; target: 0; correct: True
pred: 0; target: 0; correct: True
pred: 0; target: 0; correct: True
pred: 1; target: 1; correct: True
pred: 0; target: 0; correct: True
pred: 0; target: 0; correct: True
pred: 0; target: 0; correct: True
pred: 1; target: 1; correct: True
pred: 0; target: 0; correct: True
pred: 0; target: 0; correct: True
pred: 1; target: 1; correct: True
pred: 0; target: 0; correct: True
pred: 1; target: 1; correct: True
pred: 1; target: 1; correct: True
pred: 0; target: 0; correct: True
pred: 0; target: 1; correct: False
pred: 1; target: 1; correct: True
pred: 0; target: 0; correct: True
pred: 1; target: 0; correct: False
pred: 0; target: 0; correct: True
pred: 0; target: 1; correct: False
pred: 1; target: 1; correct: True
pred: 0; target: 0; correct: True
pred: 0; target: 0; correct: True
pred: 1; target: 1; correct: True
pred: 0; target: 0; correct: True
pred: 0; target: 0; correct: True
pred: 1; ta